# DistilBERT Fine-tune — Sentiment Analysis

**Cara pakai:** Runtime → Run all. Pilih GPU: Runtime → Change runtime type → T4 GPU.

Notebook ini:
1. Clone repo + install deps
2. Load dataset dari HuggingFace
3. Subsample stratified 30k (12k NEG / 6k NEU / 12k POS, seed=42)
4. Fine-tune DistilBERT 2 epoch (fp16, batch 32, max_len 128)
5. Evaluasi di validation split
6. Simpan model + tokenizer ke `model/final`
7. Download model sebagai zip

**Output yang harus kamu download:** `model_final.zip` → extract → isi folder `model/final/` di repo lokal.

In [ ]:
# 1. Clone repo + install
!git clone https://github.com/kusjay-space/sentiment-analysis.git
%cd sentiment-analysis
!pip install -q transformers==5.20.2 datasets==5.1.0 accelerate scikit-learn==1.9.1
print('Install selesai')

In [ ]:
# 2. Load dataset + mapping + subsample
import random
import numpy as np
import torch
from datasets import load_dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

ID2LABEL = {0: 'NEGATIVE', 1: 'NEUTRAL', 2: 'POSITIVE'}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

def map_label(label):
    if label in (0, 1): return 0  # NEGATIVE
    if label == 2:      return 1  # NEUTRAL
    return 2                       # POSITIVE

ds = load_dataset('SetFit/amazon_reviews_multi_en')

def add_sentiment(batch):
    return {'sentiment': [map_label(l) for l in batch['label']]}

train_full = ds['train'].map(add_sentiment, batched=True)
val_full   = ds['validation'].map(add_sentiment, batched=True)

def stratified_subsample(dataset, per_class, seed=SEED):
    by_class = {0: [], 1: [], 2: []}
    for i, s in enumerate(dataset['sentiment']):
        by_class[s].append(i)
    rng = random.Random(seed)
    picks = []
    for cls, idxs in by_class.items():
        rng.shuffle(idxs)
        picks.extend(idxs[:per_class[cls]])
    rng.shuffle(picks)
    return dataset.select(picks)

# 12k NEG / 6k NEU / 12k POS (mirror 2:1:2 ratio, total 30k)
train_ds = stratified_subsample(train_full, per_class={0:12000, 1:6000, 2:12000}, seed=SEED)
print(f'Train subsample: {len(train_ds)} | Val: {len(val_full)}')
print('Label dist:', {ID2LABEL[k]: sum(1 for s in train_ds['sentiment'] if s==k) for k in range(3)})

In [ ]:
# 3. Tokenisasi
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128, padding='max_length')

train_tok = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
val_tok   = val_full.map(tokenize, batched=True, remove_columns=val_full.column_names)
train_tok = train_tok.rename_column('sentiment', 'labels')
val_tok   = val_tok.rename_column('sentiment', 'labels')
train_tok.set_format('torch'); val_tok.set_format('torch')
print('Tokenisasi selesai')

In [ ]:
# 4. Model + Training
from transformers import (AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {'accuracy': accuracy_score(p.label_ids, preds),
            'f1_macro': f1_score(p.label_ids, preds, average='macro')}

use_fp16 = torch.cuda.is_available()
print(f'GPU: {torch.cuda.is_available()} | fp16: {use_fp16}')

args = TrainingArguments(
    output_dir='model/checkpoint',
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    fp16=use_fp16,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    seed=SEED,
    report_to=[],
)

trainer = Trainer(model=model, args=args, train_dataset=train_tok,
                  eval_dataset=val_tok, tokenizer=tokenizer,
                  data_collator=DataCollatorWithPadding(tokenizer),
                  compute_metrics=compute_metrics)

trainer.train()
print('Training selesai')

In [ ]:
# 5. Evaluasi validation
val_metrics = trainer.evaluate()
print('=== Validation Metrics ===')
for k, v in val_metrics.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

In [ ]:
# 6. Simpan model + tokenizer
model.save_pretrained('model/final')
tokenizer.save_pretrained('model/final')
print('Model tersimpan di model/final')
!ls -la model/final

In [ ]:
# 7. Download model
!cd model && zip -r ../model_final.zip final/
from google.colab import files
files.download('model_final.zip')
print('\n=== LANGKAH SELANJUTNYA ===')
print('1. Extract model_final.zip')
print('2. Copy isi folder final/ ke repo lokal: model/final/')
print('3. Jalankan: python -m src.inference')
print('4. Copy metrics validation di atas ke chat')